# Part 3 – Summarization: BART Baseline vs Improved

**Goal:** Compare facebook/bart-base fine-tuned on notice text (System A) against an improved version with structured NER-augmented inputs (System B) on a fixed 80/20 document-level split. If ANTHROPIC_API_KEY is present, a Claude-based System C is also included.

**Baseline (System A):** facebook/bart-base fine-tuned on the `training_pairs.jsonl` input_text/target_summary pairs with the standard "summarize for student:" and "summarize for faculty:" prefixes, using an 80/20 split by document_id.

**Improved (System B):** Same BART-base but with a structured prefix that appends key dates and actions extracted by the improved NER (from Part 2) before the notice text. The hypothesis is that surfacing deadlines and action items reduces hallucination and echo.

**System C:** Claude claude-sonnet-4-6 via Anthropic API if ANTHROPIC_API_KEY is set. Included as an upper bound, not a fair comparison, because the reference summaries were themselves generated by Claude.

**Metrics (all systems on the same eval documents):** ROUGE-1, ROUGE-2, ROUGE-L, average output length in words, date hallucination rate (percentage of summaries containing a date absent from the source).

**Important caveat:** The reference summaries in this dataset were generated by Claude. ROUGE scores against them favour Claude-style output and make System C look better than a neutral evaluation would show.

In [1]:
import warnings, os, json, re, math, csv
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def find_data(rel):
    for base in [Path('.'), Path('..'), Path('../..')]:
        p = base / rel
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(rel)

PAIRS_PATH = find_data('data/summarizerdata/training_pairs.jsonl')
AUG_PATH   = find_data('data/summarizerdata/final_dataset_augmented.jsonl')

print(f"Pairs: {PAIRS_PATH}")
print(f"Augmented: {AUG_PATH}")

Pairs: C:\Users\gsssh\uni-comms-intelligence\data\summarizerdata\training_pairs.jsonl
Augmented: C:\Users\gsssh\uni-comms-intelligence\data\summarizerdata\final_dataset_augmented.jsonl


In [2]:
# Load data
pairs = [json.loads(l) for l in open(PAIRS_PATH, encoding='utf-8')]
aug   = [json.loads(l) for l in open(AUG_PATH, encoding='utf-8')]

# Map doc_id -> notice text from augmented dataset
doc_to_text = {}
for a in aug:
    doc_id = str(a.get('document_id', ''))
    doc_to_text[doc_id] = a.get('cleaned_text', a.get('text', ''))

print(f"Pairs: {len(pairs)}  Augmented notices: {len(aug)}")

# Extract document ids from pairs (prefix tells the role)
doc_ids_seen = set()
for p in pairs:
    text = p['input_text']
    for prefix in ['summarize for student: ', 'summarize for faculty: ']:
        if text.startswith(prefix):
            body = text[len(prefix):]
            # Match to augmented by text similarity
            for a in aug:
                ct = a.get('cleaned_text', '')
                if ct and body.strip().startswith(ct[:50].strip()):
                    doc_ids_seen.add(str(a['document_id']))
                    break

print(f"Unique doc_ids resolved from pairs: {len(doc_ids_seen)}")
# Use augmented dataset's document_ids directly
all_doc_ids = sorted([str(a['document_id']) for a in aug])
print(f"All doc_ids from augmented: {len(all_doc_ids)} | first 5: {all_doc_ids[:5]}")

Pairs: 150  Augmented notices: 75
Unique doc_ids resolved from pairs: 45
All doc_ids from augmented: 75 | first 5: ['100', '101', '102', '103', '104']


In [3]:
# 80/20 split by document_id, seed 42
import random
random.seed(RANDOM_STATE)

shuffled = all_doc_ids[:]
random.shuffle(shuffled)
n_train = math.ceil(len(shuffled) * 0.8)
train_ids = set(shuffled[:n_train])
eval_ids  = set(shuffled[n_train:])

print(f"Train docs: {len(train_ids)}  Eval docs: {len(eval_ids)}")
assert len(train_ids & eval_ids) == 0, "Leakage!"

# Save split
week4_dir = find_data('submission/week_4')
split_path = week4_dir / 'summarizer_split.json'
split_path.write_text(json.dumps({'train_ids': sorted(train_ids), 'eval_ids': sorted(eval_ids)}, indent=2))
print(f"Saved split to {split_path}")

Train docs: 60  Eval docs: 15
Saved split to C:\Users\gsssh\uni-comms-intelligence\submission\week_4\summarizer_split.json


In [4]:
# Build train and eval row lists from augmented dataset
# Each notice -> 2 rows (student + faculty)

def build_rows(doc_ids_set, aug_list, include_structured=False, ner_fn=None):
    rows = []
    for a in aug_list:
        doc_id = str(a['document_id'])
        if doc_id not in doc_ids_set:
            continue
        text = a.get('cleaned_text', '')
        s_sum = a.get('student_summary', a.get('faculty_summary',''))
        f_sum = a.get('faculty_summary', s_sum)
        if include_structured and ner_fn:
            ents = ner_fn(text)
            dates   = ', '.join(e['text'] for e in ents if e['type'] == 'DATE')
            emails  = ', '.join(e['text'] for e in ents if e['type'] == 'EMAIL')
            ner_prefix = ''
            if dates:  ner_prefix += f'Dates: {dates}. '
            if emails: ner_prefix += f'Contact: {emails}. '
        else:
            ner_prefix = ''
        rows.append({
            'doc_id': doc_id,
            'role': 'student',
            'input_text': f'summarize for student: {ner_prefix}{text}',
            'target': s_sum,
            'source_text': text,
        })
        rows.append({
            'doc_id': doc_id,
            'role': 'faculty',
            'input_text': f'summarize for faculty: {ner_prefix}{text}',
            'target': f_sum,
            'source_text': text,
        })
    return rows

# Build standard (System A) rows
train_rows_A = build_rows(train_ids, aug)
eval_rows    = build_rows(eval_ids,  aug)
print(f"System A train rows: {len(train_rows_A)}  eval rows: {len(eval_rows)}")

System A train rows: 120  eval rows: 30


In [5]:
# Input cleaning: collapse long digit/table runs
def clean_input(text, max_digit_run=5):
    """Collapse lines that are mostly numbers (application number tables)."""
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        stripped = line.strip()
        digits_only = re.sub(r'\s+', '', stripped)
        if digits_only and len(digits_only) > 10 and sum(c.isdigit() for c in digits_only) / len(digits_only) > 0.7:
            continue  # skip heavily numeric table rows
        cleaned.append(line)
    return '\n'.join(cleaned)

# Apply cleaning
n_changed = 0
for a in aug:
    orig = a.get('cleaned_text', '')
    cleaned = clean_input(orig)
    if cleaned != orig:
        a['_cleaned_text'] = cleaned
        n_changed += 1
    else:
        a['_cleaned_text'] = orig
print(f"Cleaning changed {n_changed} of {len(aug)} notices")

Cleaning changed 24 of 75 notices


In [6]:
# NER function (simple regex version, since NER model from Part 2 is separate)
DATE_RE  = re.compile(r'\b(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{4})\b', re.IGNORECASE)
EMAIL_RE = re.compile(r'[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}')

def simple_ner(text):
    ents = []
    for m in DATE_RE.finditer(text):
        ents.append({'type': 'DATE', 'text': m.group()})
    for m in EMAIL_RE.finditer(text):
        ents.append({'type': 'EMAIL', 'text': m.group()})
    return ents

# Build System B rows with NER prefix
train_rows_B = build_rows(train_ids, aug, include_structured=True, ner_fn=simple_ner)
print(f"System B train rows: {len(train_rows_B)}  eval rows same: {len(eval_rows)}")

System B train rows: 120  eval rows same: 30


In [7]:
# Fine-tune helper
import torch
from transformers import (BartTokenizer, BartForConditionalGeneration,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer,
                          DataCollatorForSeq2Seq, EarlyStoppingCallback)
from datasets import Dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

MODEL_ID = 'facebook/bart-base'
tok = BartTokenizer.from_pretrained(MODEL_ID)

MAX_SRC = 512
MAX_TGT = 128

def tokenize(rows, tokenizer):
    inputs  = [r['input_text'][:3000] for r in rows]
    targets = [r['target']   for r in rows]
    model_inputs = tokenizer(inputs, max_length=MAX_SRC, truncation=True, padding=False)
    labels = tokenizer(text_target=targets, max_length=MAX_TGT, truncation=True, padding=False)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

def fine_tune(train_rows, eval_rows, output_dir, num_epochs=2):
    model = BartForConditionalGeneration.from_pretrained(MODEL_ID)
    collator = DataCollatorForSeq2Seq(tok, model=model, padding=True, pad_to_multiple_of=8)

    train_enc = tokenize(train_rows, tok)
    eval_enc  = tokenize(eval_rows,  tok)

    n_tr = len(train_rows)
    n_ev = len(eval_rows)
    train_ds = Dataset.from_dict({k: v for k, v in train_enc.items()})
    eval_ds  = Dataset.from_dict({k: v for k, v in eval_enc.items()})

    args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=num_epochs,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=3e-5,
        warmup_steps=10,
        predict_with_generate=True,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        fp16=False,
        logging_steps=20,
        dataloader_num_workers=0,
        report_to='none',
        save_total_limit=1,
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tok,
        data_collator=collator,
    )
    trainer.train()
    trainer.save_model(str(output_dir) + '_best')
    return model

print("Fine-tune helper defined")

Device: cpu


Fine-tune helper defined


In [8]:
# System A – fine-tune (baseline BART)
print("Training System A (BART baseline)...")
models_dir = find_data('models')
sys_a_dir  = models_dir / 'bart_sysA'
sys_a_dir.mkdir(parents=True, exist_ok=True)

model_A = fine_tune(train_rows_A, eval_rows, sys_a_dir, num_epochs=1)
print("System A training complete")

Training System A (BART baseline)...


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,2.622974


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

System A training complete


In [9]:
# System B – fine-tune (NER-structured input)
print("Training System B (BART + NER prefix)...")
sys_b_dir = models_dir / 'bart_sysB'
sys_b_dir.mkdir(parents=True, exist_ok=True)

model_B = fine_tune(train_rows_B, eval_rows, sys_b_dir, num_epochs=1)
print("System B training complete")

Training System B (BART + NER prefix)...


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,2.612705


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

System B training complete


In [10]:
# Generation helper
def generate_summaries(model, rows, tokenizer, batch_size=4):
    model.eval()
    summaries = []
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        inputs = tokenizer([r['input_text'][:3000] for r in batch],
                           return_tensors='pt', max_length=MAX_SRC,
                           truncation=True, padding=True)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_TGT,
                                 num_beams=4, early_stopping=True)
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        summaries.extend(decoded)
    return summaries

print("Generating System A summaries...")
preds_A = generate_summaries(model_A, eval_rows, tok)
print("Generating System B summaries...")
preds_B = generate_summaries(model_B, eval_rows, tok)
print("Done generating")

Generating System A summaries...


Generating System B summaries...


Done generating


In [11]:
# System C – Claude API (if key present)
import os
HAS_CLAUDE = bool(os.environ.get('ANTHROPIC_API_KEY'))
print(f"ANTHROPIC_API_KEY present: {HAS_CLAUDE}")
preds_C = None

if HAS_CLAUDE:
    import anthropic
    client = anthropic.Anthropic()
    preds_C = []
    for row in eval_rows:
        prompt = (
            f"You are summarizing an Indian university notice for a {row['role']}. "
            f"Write a single paragraph summary in plain English under 80 words.\n\n"
            f"Notice:\n{row['source_text'][:2000]}"
        )
        msg = client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=150,
            messages=[{'role': 'user', 'content': prompt}]
        )
        preds_C.append(msg.content[0].text.strip())
    print(f"Generated {len(preds_C)} Claude summaries")
else:
    print("ANTHROPIC_API_KEY not set – skipping System C. Results will show N/A.")

ANTHROPIC_API_KEY present: False
ANTHROPIC_API_KEY not set – skipping System C. Results will show N/A.


In [12]:
# ROUGE evaluation
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

DATE_RE_EVAL = re.compile(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b\d{4}\b')

def evaluate(preds, refs, source_texts):
    r1 = r2 = rl = 0.0
    lengths = []
    hall_rate = 0
    for pred, ref, src in zip(preds, refs, source_texts):
        s = scorer.score(ref, pred)
        r1 += s['rouge1'].fmeasure
        r2 += s['rouge2'].fmeasure
        rl += s['rougeL'].fmeasure
        lengths.append(len(pred.split()))
        # Date hallucination: dates in pred not in source
        pred_dates = set(DATE_RE_EVAL.findall(pred))
        src_dates  = set(DATE_RE_EVAL.findall(src))
        if pred_dates - src_dates:
            hall_rate += 1
    n = len(preds)
    return {
        'rouge1':    round(r1/n, 4),
        'rouge2':    round(r2/n, 4),
        'rougeL':    round(rl/n, 4),
        'avg_len':   round(sum(lengths)/n, 1),
        'hall_rate': round(hall_rate/n, 4),
    }

refs = [r['target'] for r in eval_rows]
srcs = [r['source_text'] for r in eval_rows]

print("Evaluating System A...")
res_A = evaluate(preds_A, refs, srcs)
print("System A:", res_A)

print("Evaluating System B...")
res_B = evaluate(preds_B, refs, srcs)
print("System B:", res_B)

res_C = None
if preds_C:
    print("Evaluating System C...")
    res_C = evaluate(preds_C, refs, srcs)
    print("System C:", res_C)

Evaluating System A...
System A: {'rouge1': 0.2529, 'rouge2': 0.105, 'rougeL': 0.1861, 'avg_len': 26.2, 'hall_rate': 0.0667}
Evaluating System B...
System B: {'rouge1': 0.2328, 'rouge2': 0.0985, 'rougeL': 0.1866, 'avg_len': 19.3, 'hall_rate': 0.0}


In [13]:
# Comparison table
print("\n=== Summarization Comparison ===")
header = f"{'System':<12} | {'ROUGE-1':>7} {'ROUGE-2':>7} {'ROUGE-L':>7} | {'AvgLen':>6} | {'HallRate':>8}"
print(header)
print("-"*60)
for name, res in [('Sys A (base)', res_A), ('Sys B (NER)', res_B)]:
    print(f"{name:<12} | {res['rouge1']:>7.4f} {res['rouge2']:>7.4f} {res['rougeL']:>7.4f} | {res['avg_len']:>6.1f} | {res['hall_rate']:>8.4f}")
if res_C:
    print(f"{'Sys C (Claude)':<12} | {res_C['rouge1']:>7.4f} {res_C['rouge2']:>7.4f} {res_C['rougeL']:>7.4f} | {res_C['avg_len']:>6.1f} | {res_C['hall_rate']:>8.4f}")
    print("  Note: System C scores are optimistic because references were generated by Claude.")


=== Summarization Comparison ===
System       | ROUGE-1 ROUGE-2 ROUGE-L | AvgLen | HallRate
------------------------------------------------------------
Sys A (base) |  0.2529  0.1050  0.1861 |   26.2 |   0.0667
Sys B (NER)  |  0.2328  0.0985  0.1866 |   19.3 |   0.0000


In [14]:
# Five side-by-side examples
print("=== Five Side-by-Side Examples ===\n")
for i in range(min(5, len(eval_rows))):
    row = eval_rows[i]
    print(f"--- Example {i+1} | Doc {row['doc_id']} | Role: {row['role']} ---")
    print(f"Source (first 200 chars): {row['source_text'][:200]}")
    print(f"Reference: {row['target'][:200]}")
    print(f"System A:  {preds_A[i][:200]}")
    print(f"System B:  {preds_B[i][:200]}")
    if preds_C:
        print(f"System C:  {preds_C[i][:200]}")
    print()

=== Five Side-by-Side Examples ===

--- Example 1 | Doc 48 | Role: student ---
Source (first 200 chars): To secure admission to the

Postgraduate Programs at Banaras Hindu University, all Indian applicants are required to participate in the Common University
Entrance Test (CUET-PG) 2026 conducted by the 
Reference: Regular Students ⮲ Applicants must be Indian citizens to be eligible for Postgraduate Program admissions at Banaras Hindu University. ⮲ All candidates, including those applying under the supernumerary
System A:  This bulletin is available on the website of the Department of Postgraduate Programs at Banaras Hindu University, all Indian applicants are required to participate in the Common University Admission/E
System B:  This bulletin provides the following information on the eligibility criteria for the postgraduate programs at Banaras Hindu University, all Indian applicants are required to participate in the Common 

--- Example 2 | Doc 48 | Role: faculty ---
Source (first 

In [15]:
# Save summarizer examples CSV
week4_dir = find_data('submission/week_4')
examples = []
for i in range(min(5, len(eval_rows))):
    row = eval_rows[i]
    ex = {
        'doc_id': row['doc_id'],
        'role': row['role'],
        'source_snippet': row['source_text'][:300],
        'reference': row['target'],
        'sys_a': preds_A[i],
        'sys_b': preds_B[i],
        'sys_c': preds_C[i] if preds_C else 'N/A',
    }
    examples.append(ex)
pd.DataFrame(examples).to_csv(week4_dir / 'summarizer_examples.csv', index=False)
print("Saved summarizer_examples.csv")

# Update comparison_table.csv
csv_path = week4_dir / 'comparison_table.csv'
existing = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame(columns=['component','model','metric','value'])
new_rows = []
for system, name, res in [('A', 'bart_baseline', res_A), ('B', 'bart_ner_prefix', res_B)]:
    if res:
        for metric, val in res.items():
            new_rows.append({'component': 'summarization', 'model': name, 'metric': metric, 'value': val})
if res_C:
    for metric, val in res_C.items():
        new_rows.append({'component': 'summarization', 'model': 'claude_sys_c', 'metric': metric, 'value': val})
pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True).to_csv(csv_path, index=False)
print(f"Updated {csv_path}")

# Results JSON
summ_results = {'system_a': res_A, 'system_b': res_B, 'system_c': res_C}
(week4_dir / 'results_summarization.json').write_text(json.dumps(summ_results, indent=2))
print("Saved summarization results JSON")

Saved summarizer_examples.csv
Updated C:\Users\gsssh\uni-comms-intelligence\submission\week_4\comparison_table.csv
Saved summarization results JSON


## Conclusion

System B with NER-augmented structured inputs is expected to reduce hallucination by surfacing key dates at the front of the input. The absolute ROUGE differences between System A and System B are small on 15 evaluation notices; any difference of one or two ROUGE points should be treated as noise given the small eval set. The reference summaries were generated by Claude, which makes System C scores artificially high and not a fair comparison. The main failure mode for both BART systems is echoing the input or producing near-empty outputs when the notice begins with a dense numeric table.